# Pipeline — Method 3b
**Electricity: CO2online Stromspiegel 2025 (Without Electric Water Heating)**

This notebook builds `DEA_method3b_final.parquet` from the raw OSM building dataset.
It calculates electricity demand, heat demand, and PV potential for all 4.1M NRW residential buildings.

| Step | What it does |
|------|--------------|
| 1 | Imports and configuration |
| 2 | CO2online lookup table |
| 3 | Census 2022 TABULA statistics |
| 4 | Load raw building data |
| 5 | Footprint area |
| 6 | Living area (EnEV 2009) |
| 7 | Electricity demand (Method 3b) |
| 8 | PV yield from PVGIS API |
| 9 | Zone assignment |
| 10 | PV potential |
| 11 | Heat demand (TABULA) |
| 12 | Save parquet |

**Note:** Steps 1-12 take 10-15 minutes to run on the full 4.1M building dataset.

---
## Step 1 — Imports and Configuration

In [35]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from pyproj import Transformer

# Relative paths — works on any PC regardless of username
BASE_DIR    = os.getcwd()
INPUT_FILE  = os.path.join(BASE_DIR, 'DEA_combined.parquet')
HEAT_FILE   = os.path.join(BASE_DIR, 'final_tabula.xlsx')
OUTPUT_FILE = os.path.join(BASE_DIR, 'DEA_method3b_final.parquet')

print('Imports OK')
print(f'Input  : {INPUT_FILE}')
print(f'Heat   : {HEAT_FILE}')
print(f'Output : {OUTPUT_FILE}')

Imports OK
Input  : c:\Users\zaito\Downloads\clustering\DEA_combined.parquet
Heat   : c:\Users\zaito\Downloads\clustering\final_tabula.xlsx
Output : c:\Users\zaito\Downloads\clustering\DEA_method3b_final.parquet


In [36]:
# Building geometry constants
ROOF_CORRECTION   = 3.0   # metres subtracted from building height for roof
MIN_HEATED_HEIGHT = 3.5   # minimum heated height in metres

# PV system parameters
ROOF_FRACTION    = 0.20   # fraction of footprint area used for PV
PANEL_EFFICIENCY = 0.20   # 20% panel efficiency (crystalline silicon)
TILT             = 35     # panel tilt in degrees
AZIMUTH          = -45    # panel azimuth (-45 = south-west, 0 = south)
SYSTEM_LOSS      = 14     # system losses in percent (PVGIS default)

# Two PVGIS reference locations covering NRW
ESSEN          = {'name': 'Essen',          'lat': 51.455, 'lon': 7.011}
BAD_MARIENBERG = {'name': 'Bad Marienberg', 'lat': 50.651, 'lon': 7.952}

SIZE_CLASSES = ['SFH', 'TH', 'MFH', 'AB']
print('Configuration ready')

Configuration ready


---
## Step 2 — CO2online Lookup Table

Electricity demand per dwelling per year by household size and building category.

**Source:** CO2online Stromspiegel via Statista (Mai 2025)
**Values:** WITHOUT electric water heating (Ohne elektrischer Warmwasserbereitung)

**Why without electric heating:**
SESMG optimizes domestic hot water heating separately.
Including it here would double-count the electricity demand in the simulation.

**Note on 4-person data:** From April 2023 — most recent available on Statista at time of analysis.

**Category rule:**
- SFH and TH → EFH column (single-unit dwellings)
- MFH and AB → MFH column (apartment dwellings, smaller per-unit demand)

In [37]:
# CO2online Stromspiegel 2025 — WITHOUT electric water heating
# Key: persons (1-5), Value: (EFH kWh/yr, MFH/AB kWh/yr)
ELEC_BY_PERSONS = {
    1: (1_800, 1_200),   # Statista Mai 2025
    2: (2_700, 1_900),   # Statista Mai 2025
    3: (3_500, 2_400),   # Statista Mai 2025
    4: (4_000, 2_900),   # Statista April 2023 (latest available)
    5: (4_500, 3_100),   # Statista Mai 2025
}

# Print table for reference
print('CO2online Stromspiegel 2025 — Without Electric Water Heating')
print(f'  {"Persons":<10} {"SFH/TH (kWh/yr)":<20} {"MFH/AB (kWh/yr)"}')
print('-' * 50)
for p, (efh, mfh) in ELEC_BY_PERSONS.items():
    print(f'  {p:<10} {efh:<20,} {mfh:,}')

CO2online Stromspiegel 2025 — Without Electric Water Heating
  Persons    SFH/TH (kWh/yr)      MFH/AB (kWh/yr)
--------------------------------------------------
  1          1,800                1,200
  2          2,700                1,900
  3          3,500                2,400
  4          4,000                2,900
  5          4,500                3,100


---
## Step 3 — Census 2022 TABULA Statistics

Average dwelling size and average number of inhabitants per tabula type.

**Source:** German Census 2022 (Destatis), filtered to homogeneous 100m grid cells
where only one building type exists — ensures averages are not distorted by mixed neighbourhoods.

These two values are used in Step 7 to calculate:
- Number of dwellings per building (Al / avg_dwelling_size)
- Number of persons per dwelling (avg_inhabitants)


In [38]:
TABULA_STATS = pd.DataFrame([
    # Apartment blocks (AB)
    ('DE.N.AB.02',  63, 1), ('DE.N.AB.03',  88, 2), ('DE.N.AB.04',  69, 2),
    ('DE.N.AB.05',  69, 2), ('DE.N.AB.06',  69, 2), ('DE.N.AB.07',  70, 2),
    ('DE.N.AB.08',  64, 2), ('DE.N.AB.09',  69, 2), ('DE.N.AB.10',  68, 2),
    ('DE.N.AB.11',  58, 1), ('DE.N.AB.12',  62, 2),
    # Multi-family homes (MFH)
    ('DE.N.MFH.01', 80, 2), ('DE.N.MFH.02', 82, 2), ('DE.N.MFH.03', 73, 2),
    ('DE.N.MFH.04', 74, 2), ('DE.N.MFH.05', 74, 2), ('DE.N.MFH.06', 74, 2),
    ('DE.N.MFH.07', 79, 2), ('DE.N.MFH.08', 78, 2), ('DE.N.MFH.09', 79, 2),
    ('DE.N.MFH.10', 84, 2), ('DE.N.MFH.11', 76, 2), ('DE.N.MFH.12', 77, 2),
    # Single-family homes (SFH)
    ('DE.N.SFH.01', 118, 2), ('DE.N.SFH.02', 119, 2), ('DE.N.SFH.03', 116, 2),
    ('DE.N.SFH.04', 123, 2), ('DE.N.SFH.05', 123, 2), ('DE.N.SFH.06', 123, 2),
    ('DE.N.SFH.07', 128, 2), ('DE.N.SFH.08', 127, 2), ('DE.N.SFH.09', 129, 2),
    ('DE.N.SFH.10', 135, 3), ('DE.N.SFH.11', 142, 3), ('DE.N.SFH.12', 141, 3),
    # Terraced houses (TH)
    ('DE.N.TH.01',   99, 2), ('DE.N.TH.02', 100, 2), ('DE.N.TH.03',  99, 2),
    ('DE.N.TH.04',  107, 2), ('DE.N.TH.05', 107, 2), ('DE.N.TH.06', 107, 2),
    ('DE.N.TH.07',  117, 2), ('DE.N.TH.08', 117, 2), ('DE.N.TH.09', 121, 3),
    ('DE.N.TH.10',  124, 3), ('DE.N.TH.11', 116, 3), ('DE.N.TH.12', 114, 3),
], columns=['tabula_key', 'avg_dwelling_size_m2', 'avg_inhabitants'])

# Fallback values for tabula types not in the table
FALLBACK_SIZE = {'SFH': 120, 'TH': 108, 'MFH': 74, 'AB': 69}
FALLBACK_INH  = {'SFH': 2,   'TH': 2,   'MFH': 2,  'AB': 2}

print(f'TABULA_STATS loaded: {len(TABULA_STATS)} tabula types')
print(TABULA_STATS.head(5).to_string(index=False))

TABULA_STATS loaded: 47 tabula types
tabula_key  avg_dwelling_size_m2  avg_inhabitants
DE.N.AB.02                    63                1
DE.N.AB.03                    88                2
DE.N.AB.04                    69                2
DE.N.AB.05                    69                2
DE.N.AB.06                    69                2


---
## Step 4 — Load Raw Building Data

The raw input is `DEA_combined.parquet` — the OSM building dataset for NRW. It contains building footprint geometry, height,
tabula type, refurbishment state, and construction year.

The CRS (coordinate reference system) is detected automatically from the geometry coordinates.

In [39]:
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f'Cannot find: {INPUT_FILE}\nMake sure DEA_combined.parquet is in the same folder as this notebook.')

gdf = gpd.read_parquet(INPUT_FILE)
print(f'Buildings       : {len(gdf):,}')
print(f'Geometry column : {gdf.geometry.name}')
print(f'Columns         : {gdf.columns.tolist()}')

cx_raw = gdf.geometry.centroid.x.values
cy_raw = gdf.geometry.centroid.y.values
NATIVE_EPSG = 3035 if cx_raw.min() > 1_000_000 else 25832
print(f'Detected CRS    : EPSG:{NATIVE_EPSG}')
print(f'tabula_type eg  : {gdf["tabula_type"].iloc[0]}')

Buildings       : 4,133,323
Geometry column : footprint
Columns         : ['footprint', 'height_m', 'id', 'construction_year', 'size_class', 'refurbishment_state', 'tabula_type']
Detected CRS    : EPSG:3035
tabula_type eg  : DE.N.TH.05.Gen.ReEx.001.003


---
## Step 5 — Footprint Area

Calculated directly from the building polygon geometry.
This is the ground floor area of the building in square metres.

In [40]:
gdf['area_m2'] = gdf.geometry.area
print(f'Footprint area:')
print(f'  min : {gdf["area_m2"].min():.1f} m2')
print(f'  mean: {gdf["area_m2"].mean():.1f} m2')
print(f'  max : {gdf["area_m2"].max():.1f} m2')

Footprint area:
  min : 1.0 m2
  mean: 123.6 m2
  max : 20950.3 m2


---
## Step 6 — Living Area (EnEV 2009 / Loga et al. 2012)

The conditioned living area Al is derived from building geometry:

```
heated_height = max(h_building - 3.0,  3.5)   # subtract roof, minimum 3.5m
volume_m3     = area_m2 × heated_height
Au_m2         = 0.32 × volume_m3               # EnEV 2009
Al_m2         = Au_m2 / 1.3                    # Loga et al. 2012
```

`ROOF_CORRECTION = 3.0m` subtracts the non-heated roof space from building height.
`MIN_HEATED_HEIGHT = 3.5m` prevents unrealistically small values for very short buildings.

`This Al will be used in step 7.


In [41]:
gdf['heated_height'] = (gdf['height_m'] - ROOF_CORRECTION).clip(lower=MIN_HEATED_HEIGHT)
gdf['volume_m3']     = gdf['area_m2'] * gdf['heated_height']
gdf['Au_m2']         = 0.32 * gdf['volume_m3']
gdf['Al_m2']         = gdf['Au_m2'] / 1.3

print(f'Living area (Al):')
print(f'  min : {gdf["Al_m2"].min():.1f} m2')
print(f'  mean: {gdf["Al_m2"].mean():.1f} m2')
print(f'  max : {gdf["Al_m2"].max():.1f} m2')

Living area (Al):
  min : 0.9 m2
  mean: 220.5 m2
  max : 51288.5 m2


---
## Step 7 — Electricity Demand (Method 3b)

Four sub-steps:

**7.1 — Tabula key extraction**
Extracts the 4-part tabula key from the full tabula_type string.
Example: `DE.N.TH.05.Gen.ReEx.001.003` → `DE.N.TH.05`

**7.2 — Merge Census 2022 stats**
Joins average dwelling size and average inhabitants per tabula type.

**7.3 — Number of dwellings**
```
n_dwellings = max(1, round(Al / avg_dwelling_size))
```
Minimum 1 — every building has at least one dwelling.

**7.4 — Electricity lookup**
```
persons  = max(1, min(5, round(avg_inhabitants)))
per_dw   = CO2online[persons][EFH or MFH]
elec_m3b = n_dwellings × per_dw
```

In [42]:
# 7.1 Extract 4-part tabula key
gdf['tabula_key'] = gdf['tabula_type'].str.split('.').str[:4].str.join('.')
print(f'tabula_key sample: {gdf["tabula_key"].unique()[:4].tolist()}')

tabula_key sample: ['DE.N.TH.05', 'DE.N.TH.04', 'DE.N.SFH.05', 'DE.N.MFH.06']


In [43]:
# 7.2 Merge Census 2022 stats
gdf = gdf.merge(TABULA_STATS, on='tabula_key', how='left')

missing = gdf['avg_dwelling_size_m2'].isna()
gdf.loc[missing, 'avg_dwelling_size_m2'] = gdf.loc[missing, 'size_class'].map(FALLBACK_SIZE)
gdf.loc[missing, 'avg_inhabitants']      = gdf.loc[missing, 'size_class'].map(FALLBACK_INH)

print(f'Matched via TABULA_STATS : {(~missing).sum():,} buildings')
print(f'Using fallback averages  : {missing.sum():,} buildings')

Matched via TABULA_STATS : 4,133,323 buildings
Using fallback averages  : 0 buildings


In [ ]:
# 7.3 Number of dwellings per building
gdf['n_dwellings_m3b'] = (
    gdf['Al_m2'] / gdf['avg_dwelling_size_m2']
).round().clip(lower=1).astype(int)

# SFH and TH always have exactly 1 dwelling by definition
is_efh = gdf['size_class'].isin(['SFH', 'TH'])
gdf.loc[is_efh, 'n_dwellings_m3b'] = 1

print(f'Dwelling count distribution (all buildings):')
print(gdf['n_dwellings_m3b'].value_counts().sort_index().head(10).to_string())
print()
print('SFH dwelling counts (must be all 1):')
print(gdf[gdf['size_class']=='SFH']['n_dwellings_m3b'].value_counts().to_dict())
print('TH  dwelling counts (must be all 1):')
print(gdf[gdf['size_class']=='TH']['n_dwellings_m3b'].value_counts().to_dict())

Dwelling count distribution:
n_dwellings_m3b
1     3325724
2      122928
3      125232
4      119160
5      104935
6       89063
7       66174
8       45109
9       30964
10      21618


In [45]:
# 7.4 CO2online lookup — vectorised NumPy indexing (much faster than apply)
persons = gdf['avg_inhabitants'].round().clip(lower=1, upper=5).astype(int)

EFH_KWH = np.array([0, 1_800, 2_700, 3_500, 4_000, 4_500])  # index = persons
MFH_KWH = np.array([0, 1_200, 1_900, 2_400, 2_900, 3_100])

is_efh = gdf['size_class'].isin(['SFH', 'TH'])
gdf['demand_per_dw_m3b'] = np.where(is_efh, EFH_KWH[persons], MFH_KWH[persons])
gdf['elec_m3b_kwh']      = gdf['n_dwellings_m3b'] * gdf['demand_per_dw_m3b']

print(f'Electricity demand (Method 3b):')
print(f'  min  : {gdf["elec_m3b_kwh"].min():>12,.0f} kWh/yr')
print(f'  mean : {gdf["elec_m3b_kwh"].mean():>12,.0f} kWh/yr')
print(f'  max  : {gdf["elec_m3b_kwh"].max():>12,.0f} kWh/yr')
print(f'  total: {gdf["elec_m3b_kwh"].sum()/1e9:.3f} TWh/yr')
print(f'  BDEW benchmark: 31.96 TWh/yr')
print(f'  Difference    : {(gdf["elec_m3b_kwh"].sum()/1e9 - 31.96)/31.96*100:.1f}%')
print(f'\nBy size class:')
for sc in SIZE_CLASSES:
    sub = gdf[gdf['size_class'] == sc]
    print(f'  {sc}: mean={sub["elec_m3b_kwh"].mean():>10,.0f}  median={sub["elec_m3b_kwh"].median():>10,.0f}  n={len(sub):>9,}')

Electricity demand (Method 3b):
  min  :        1,200 kWh/yr
  mean :        4,484 kWh/yr
  max  :    1,316,700 kWh/yr
  total: 18.533 TWh/yr
  BDEW benchmark: 31.96 TWh/yr
  Difference    : -42.0%

By size class:
  SFH: mean=     2,728  median=     2,700  n=2,723,098
  TH: mean=     2,786  median=     2,700  n=  507,633
  MFH: mean=    10,508  median=     7,600  n=  897,247
  AB: mean=    49,118  median=    34,200  n=    5,345


---
## Step 8 — PV Yield from PVGIS API

Fetches annual PV yield in kWh per kWp for two NRW reference locations.
PVGIS (Photovoltaic Geographical Information System) is the EU's official
solar radiation database.

Two locations cover the range of solar radiation across NRW:
- **Essen** (north-west NRW) — lower radiation
- **Bad Marienberg** (south-east NRW) — higher radiation

If the API call fails, fallback values are used:
- Essen: 967.4 kWh/kWp/yr
- Bad Marienberg: 952.5 kWh/kWp/yr

In [46]:
def get_pvgis_yield(lat, lon, tilt, azimuth, loss):
    url = 'https://re.jrc.ec.europa.eu/api/v5_3/PVcalc'
    params = {
        'lat': lat, 'lon': lon, 'peakpower': 1,
        'pvtechchoice': 'crystSi', 'loss': loss,
        'angle': tilt, 'aspect': azimuth,
        'outputformat': 'json', 'raddatabase': 'PVGIS-SARAH3',
    }
    try:
        r = requests.get(url, params=params, timeout=30)
        if r.status_code != 200:
            print(f'  WARNING: {r.status_code}')
            return None
        val = r.json()['outputs']['totals']['fixed']['E_y']
        print(f'  -> {val:.1f} kWh/kWp/yr  [PVGIS API]')
        return val
    except Exception as e:
        print(f'  WARNING: {e}')
        return None

print(f'Essen ({ESSEN["lat"]}N, {ESSEN["lon"]}E):')
yield_essen = get_pvgis_yield(ESSEN['lat'], ESSEN['lon'], TILT, AZIMUTH, SYSTEM_LOSS)

print(f'Bad Marienberg ({BAD_MARIENBERG["lat"]}N, {BAD_MARIENBERG["lon"]}E):')
yield_bm = get_pvgis_yield(BAD_MARIENBERG['lat'], BAD_MARIENBERG['lon'], TILT, AZIMUTH, SYSTEM_LOSS)

if yield_essen is None:
    yield_essen = 967.4
    print(f'  Fallback: {yield_essen}')
if yield_bm is None:
    yield_bm = 952.5
    print(f'  Fallback: {yield_bm}')

print(f'\nEssen          : {yield_essen:.1f} kWh/kWp/yr')
print(f'Bad Marienberg : {yield_bm:.1f} kWh/kWp/yr')

Essen (51.455N, 7.011E):
  -> 967.4 kWh/kWp/yr  [PVGIS API]
Bad Marienberg (50.651N, 7.952E):
  -> 952.5 kWh/kWp/yr  [PVGIS API]

Essen          : 967.4 kWh/kWp/yr
Bad Marienberg : 952.5 kWh/kWp/yr


---
## Step 9 — Zone Assignment

Each building is assigned to its nearest reference city based on Euclidean distance
in the projected coordinate system (EPSG:3035 or EPSG:25832).

This determines which PVGIS yield value the building gets:
- Essen zone → 967.4 kWh/kWp/yr
- Bad Marienberg zone → 952.5 kWh/kWp/yr

In [47]:
transformer = Transformer.from_crs('EPSG:4326', f'EPSG:{NATIVE_EPSG}', always_xy=True)
essen_x, essen_y = transformer.transform(ESSEN['lon'],          ESSEN['lat'])
bm_x,    bm_y    = transformer.transform(BAD_MARIENBERG['lon'], BAD_MARIENBERG['lat'])

dist_essen = np.sqrt((cx_raw - essen_x)**2 + (cy_raw - essen_y)**2)
dist_bm    = np.sqrt((cx_raw - bm_x)**2    + (cy_raw - bm_y)**2)

gdf['nearest_city']   = np.where(dist_bm < dist_essen, 'Bad Marienberg', 'Essen')
gdf['specific_yield'] = np.where(dist_bm < dist_essen, yield_bm, yield_essen)

n_e  = (gdf['nearest_city'] == 'Essen').sum()
n_bm = (gdf['nearest_city'] == 'Bad Marienberg').sum()
print(f'Essen          : {n_e:,}  ({n_e/len(gdf)*100:.1f}%)  @ {yield_essen:.1f} kWh/kWp/yr')
print(f'Bad Marienberg : {n_bm:,}  ({n_bm/len(gdf)*100:.1f}%)  @ {yield_bm:.1f} kWh/kWp/yr')

Essen          : 3,533,493  (85.5%)  @ 967.4 kWh/kWp/yr
Bad Marienberg : 599,830  (14.5%)  @ 952.5 kWh/kWp/yr


---
## Step 10 — PV Potential

```
pv_roof_area   = area_m2 × 0.20        (20% of footprint area)
pv_peak_power  = pv_roof_area × 0.20   (20% panel efficiency)
pv_potential   = pv_peak_power × specific_yield
ss_m3b         = pv_potential / elec_m3b
```

The 20% roof fraction was selected by testing four scenarios (16%, 20%, 25%, 50%)
and choosing the fraction that gave the most plausible self-sufficiency result
relative to the literature benchmark (Weniger et al. 2014).

**Note:** This PV calculation will be replaced by the LOD2 roof area clustering
for the actual SESMG input. The 20% fraction is used here for the self-sufficiency
validation comparison between Method 1 and Method 3b only.

In [48]:
gdf['pv_roof_area_m2']   = gdf['area_m2'] * ROOF_FRACTION
gdf['pv_peak_power_kwp'] = gdf['pv_roof_area_m2'] * PANEL_EFFICIENCY
gdf['pv_potential_kwh']  = gdf['pv_peak_power_kwp'] * gdf['specific_yield']
gdf['ss_m3b']            = gdf['pv_potential_kwh'] / gdf['elec_m3b_kwh']

print(f'PV total : {gdf["pv_potential_kwh"].sum()/1e9:.3f} TWh/yr')
print(f'PV mean  : {gdf["pv_potential_kwh"].mean():,.0f} kWh/yr')
print(f'SS mean  : {gdf["ss_m3b"].mean():.2f}x   median: {gdf["ss_m3b"].median():.2f}x')
print(f'SS >= 1.0: {(gdf["ss_m3b"] >= 1).mean()*100:.1f}% of buildings')

PV total : 19.725 TWh/yr
PV mean  : 4,772 kWh/yr
SS mean  : 1.39x   median: 1.19x
SS >= 1.0: 63.7% of buildings


---
## Step 11 — Heat Demand (TABULA × Al)

Heat demand is identical between Method 1 and Method 3b — same formula, same source.

```
heat_total_kwh = (heat_EUI + DHW_EUI) × Al_m2
```

Where:
- `heat_EUI` — specific heat demand from TABULA per tabula type per refurbishment state
- `DHW_EUI` — domestic hot water demand from TABULA
- `Al_m2` — conditioned living area from Step 6

Each building gets its own heat value based on its actual assigned refurbishment state
(State 1 = original, State 2 = partial refurbishment, State 3 = advanced refurbishment).

In [49]:
heat_raw  = pd.read_excel(HEAT_FILE, sheet_name='Germany')
heat_long = heat_raw.melt(
    id_vars=['tabula_type', 'domestic hot water system'],
    value_vars=['existing state (Heating)', 'Usual refurbishment', 'advanced refurbishment'],
    var_name='refurb_col', value_name='heat_eui'
)
refurb_map = {
    'existing state (Heating)': '1',
    'Usual refurbishment':      '2',
    'advanced refurbishment':   '3'
}
heat_long['refurbishment_state'] = heat_long['refurb_col'].map(refurb_map)
heat_long = heat_long.rename(columns={
    'tabula_type':              'tabula_key',
    'domestic hot water system':'dhw_eui'
})
print(f'TABULA heat data loaded: {len(heat_long)} rows')

TABULA heat data loaded: 141 rows


In [50]:
gdf['tabula_key']              = gdf['tabula_type'].str.split('.').str[:4].str.join('.')
gdf['refurbishment_state_str'] = gdf['refurbishment_state'].astype(str)

gdf = gdf.merge(
    heat_long[['tabula_key','refurbishment_state','heat_eui','dhw_eui']],
    left_on=['tabula_key','refurbishment_state_str'],
    right_on=['tabula_key','refurbishment_state'],
    how='left',
    suffixes=('','_heat')
)
gdf['heat_total_kwh'] = (gdf['heat_eui'] + gdf['dhw_eui']) * gdf['Al_m2']

print(f'Matched  : {gdf["heat_total_kwh"].notna().sum():,} buildings')
print(f'Missing  : {gdf["heat_total_kwh"].isna().sum():,} buildings')
print(f'Heat total: {gdf["heat_total_kwh"].sum()/1e9:.2f} TWh/yr')
print(f'Heat mean : {gdf["heat_total_kwh"].mean():,.0f} kWh/yr')
print(f'\nNote: heat is ~4x electricity — expected for Germany')

Matched  : 4,133,323 buildings
Missing  : 0 buildings
Heat total: 135.39 TWh/yr
Heat mean : 32,755 kWh/yr

Note: heat is ~4x electricity — expected for Germany


---
## Step 12 — Save Parquet

Saves the final output as `DEA_method3b_final.parquet`.
Only the columns needed for clustering are kept.

In [51]:
keep = ['footprint', 'id', 'height_m', 'size_class', 'tabula_type',
        'refurbishment_state', 'construction_year',
        'area_m2', 'heated_height', 'volume_m3', 'Au_m2', 'Al_m2',
        'tabula_key', 'avg_dwelling_size_m2', 'avg_inhabitants',
        'n_dwellings_m3b', 'demand_per_dw_m3b', 'elec_m3b_kwh',
        'heat_total_kwh',
        'nearest_city', 'specific_yield',
        'pv_roof_area_m2', 'pv_peak_power_kwp', 'pv_potential_kwh',
        'ss_m3b']

out = gdf[[c for c in keep if c in gdf.columns]]
if 'footprint' in out.columns:
    out = gpd.GeoDataFrame(out, geometry='footprint', crs=gdf.crs)
out.to_parquet(OUTPUT_FILE, index=False)

print(f'Saved {len(out):,} buildings -> {OUTPUT_FILE}')
print(f'\nOutput columns:')
print('  elec_m3b_kwh         — Method 3b electricity demand [kWh/yr]')
print('  n_dwellings_m3b      — Dwelling count from Census 2022')
print('  demand_per_dw_m3b    — kWh/dwelling from CO2online lookup')
print('  avg_dwelling_size_m2 — Census 2022 avg dwelling size per tabula_type')
print('  avg_inhabitants      — Census 2022 avg persons per dwelling')
print('  heat_total_kwh       — Heat demand from TABULA [kWh/yr]')
print('  pv_potential_kwh     — Annual PV generation potential [kWh/yr]')
print('  ss_m3b               — Self-sufficiency: PV / elec_m3b')
print('\nDone.')

Saved 4,133,323 buildings -> c:\Users\zaito\Downloads\clustering\DEA_method3b_final.parquet

Output columns:
  elec_m3b_kwh         — Method 3b electricity demand [kWh/yr]
  n_dwellings_m3b      — Dwelling count from Census 2022
  demand_per_dw_m3b    — kWh/dwelling from CO2online lookup
  avg_dwelling_size_m2 — Census 2022 avg dwelling size per tabula_type
  avg_inhabitants      — Census 2022 avg persons per dwelling
  heat_total_kwh       — Heat demand from TABULA [kWh/yr]
  pv_potential_kwh     — Annual PV generation potential [kWh/yr]
  ss_m3b               — Self-sufficiency: PV / elec_m3b

Done.
